# 第 6 章 · 自定义接口与动作

本章回答：自己的消息格式怎么定义？`.msg` 里能写哪些类型？自定义接口为什么要单独放一个 CMake 包？动作服务端 / 客户端怎么写？动作底层到底是什么？

配套讲义：仓库里 `ros2/notes/lessons/06-interfaces-actions.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 rosidl 和 rclpy。这里用一个小解析器按接口文档的规则拆 .msg 行，用一个字典写下讲义总图里的目标状态机，只走"成功"和"服务端放弃"两条路（取消那条留给练习 6-3）。

In [ ]:
import re

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 数组 | `[]` / `[5]` / `[<=5]` | `parse_field(...)` | 不限长度 / 正好 5 个 / 最多 5 个 |
| 第 1 步 默认值与常量 | 第三项是默认值；`=` 是常量 | `parse_field('int32 X=123')` | 常量 123 |
| 总图 动作底层 | 3 个服务 + 2 个话题 | `action_parts('/fibonacci')` | `/fibonacci/_action/send_goal` 等 5 个 |
| 第 3 步 Fibonacci | order = 5，每项发反馈 | `fibonacci_execute(5)` | 反馈 [0,1,1] 起；结果 [0,1,1,2,3,5] |
| 第 3 步 状态机 | ACCEPTED → EXECUTING → 终态 | `run_goal([...])` | SUCCEEDED / ABORTED |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**函数 `action_parts`**：一个动作在底层 = 3 个服务 + 2 个话题，名字都藏在 _action/ 下面。

In [ ]:
def action_parts(action: str) -> list[tuple[str, str]]:
    """一个动作在底层 = 3 个服务 + 2 个话题，名字都藏在 _action/ 下面。"""
    base = f"{action}/_action"
    return [(f"{base}/send_goal", "服务"), (f"{base}/cancel_goal", "服务"), (f"{base}/get_result", "服务"),
            (f"{base}/feedback", "话题"), (f"{base}/status", "话题")]

**函数 `fibonacci_execute`**：官方 server 的 execute_callback：每算出一项发一次反馈，最后返回整列。

In [ ]:
def fibonacci_execute(order: int) -> tuple[list[list[int]], list[int]]:
    """官方 server 的 execute_callback：每算出一项发一次反馈，最后返回整列。"""
    sequence = [0, 1]
    feedbacks = []
    for i in range(1, order):
        sequence.append(sequence[i] + sequence[i - 1])
        feedbacks.append(list(sequence))
    return feedbacks, sequence

**常量 `LINE`**

In [ ]:
LINE = re.compile(
    r"^(?P<type>[\w/]+?)(?P<bound_str><=\d+)?(?P<array>\[(?P<cap><=)?(?P<n>\d*)\])?\s+"
    r"(?P<name>\w+)(?:\s*=\s*(?P<const>.+)|\s+(?P<default>.+))?$"
)

**函数 `parse_field`**：一行 "类型 名字 [默认值]" 或 "类型 名字=常量"，拆成字典。

In [ ]:
def parse_field(line: str) -> dict:
    """一行 "类型 名字 [默认值]" 或 "类型 名字=常量"，拆成字典。"""
    m = LINE.match(line.split("#")[0].strip())
    if not m:
        raise ValueError(line)
    d = m.groupdict()
    out = {"类型": d["type"], "名字": d["name"]}
    if d["bound_str"]:
        out["字符串上限"] = int(d["bound_str"][2:])
    if d["array"]:
        n = d["n"]
        out["数组"] = f"最多 {n} 个" if d["cap"] else (f"正好 {n} 个" if n else "不限长度")
    if d["const"] is not None:
        out["常量"] = d["const"].strip('"')
    elif d["default"] is not None:
        out["默认值"] = d["default"].strip('"')
    return out

**常量 `TERMINAL`**

In [ ]:
TERMINAL = {"SUCCEEDED", "ABORTED", "CANCELED"}

**常量 `TRANSITIONS`**

In [ ]:
TRANSITIONS = {  # 讲义总图：目标状态机，(当前状态, 事件) → 下一个状态
    ("ACCEPTED", "execute"): "EXECUTING",
    ("EXECUTING", "succeed"): "SUCCEEDED",
    ("EXECUTING", "abort"): "ABORTED",
    ("ACCEPTED", "cancel_goal"): "CANCELING",
    ("EXECUTING", "cancel_goal"): "CANCELING",
    ("CANCELING", "canceled"): "CANCELED",
}

**函数 `run_goal`**：从 ACCEPTED 出发，按事件一路走；走到终态就停。不合法的事件直接报错。

In [ ]:
def run_goal(events: list[str]) -> list[str]:
    """从 ACCEPTED 出发，按事件一路走；走到终态就停。不合法的事件直接报错。"""
    state, path = "ACCEPTED", ["ACCEPTED"]
    for ev in events:
        if state in TERMINAL:
            break
        state = TRANSITIONS[(state, ev)]
        path.append(state)
    return path

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · .msg 每行是\"类型 名字\"，后面可以跟默认值或常量")
for line in ["int32[] unbounded_integer_array", "int32[5] five_integers_array",
             "int32[<=5] up_to_five_integers_array", "string<=10 up_to_ten_characters",
             "uint8 x 42", 'string full_name "John Doe"', "int32 X=123",
             "geometry_msgs/Point center"]:
    print(f"  {line:<38} → {parse_field(line)}")

In [ ]:
print("\n总图 · 动作 /fibonacci 在底层是这 5 个名字")
for name, kind in action_parts("/fibonacci"):
    print(f"  {kind}  {name}")

In [ ]:
print("\n第三步 · Fibonacci 动作：order=5，每算一项发一次反馈")
feedbacks, result = fibonacci_execute(5)
for fb in feedbacks:
    print("  反馈:", fb)
print("  结果:", result)

In [ ]:
print("\n第三步（续）· 目标状态机：成功和服务端放弃两条路")
print("  ", " → ".join(run_goal(["execute", "succeed"])))
print("  ", " → ".join(run_goal(["execute", "abort"])))

## 3. 代码里才看得到的

1. **反馈要拷一份**：`feedbacks.append(list(sequence))`。直接 append `sequence` 的话，后面的修改会改到之前存的每一条——Python 列表是引用。
2. **状态机里取消那条路（CANCELING → CANCELED）写进了字典，但 `main()` 没走**，留给练习 6-3。
3. **解析器只认讲义里那几种写法**，真 rosidl 还支持 `wstring`、嵌套包名的完整写法等。

## 4. 自己改一改

- 把 `order` 改成 8，看结果数列和反馈条数；
- 给 `parse_field` 喂一行 `float64[3] position`，看它怎么拆。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **写接口**：一行"类型 名字"；`[]` 不限长、`[N]` 定长、`[<=N]` 有上限；默认值放第三项，常量全大写用等号。
- **接口包**：只能 ament_cmake；CMake 里 `rosidl_generate_interfaces`，`package.xml` 三行（生成器、运行时、组成员）。
- **动作**：三服务（发目标、取消、取结果）两话题（反馈、状态）；三中间态三终态。
- **标准优先**：能用 `geometry_msgs` / `sensor_msgs` 就别自定义。